# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

Everything before this notebook was validation. This one turns the validated Week-5/6 model and
the Week-4 rule into something a human content team can actually pick up: a ranked queue, plain
reasons, an archetype view, a cost/value read, and — just as important — the limits, the no-go
list, and what would tell us this playbook has gone stale.

Skills used: `writing-honest-claims` + `flyrank/flyrank-data` (per `skills/README.md`).

## 1. Ranked actions + reason codes

**Design choice:** the deterministic Week-4/8 rule (`ctr_gap_ratio` vs. each page's own
position-tier benchmark) stays the **primary, auditable driver** of the ranked queue — it's the
one a reviewer can explain in one sentence. The Week-5/6 logistic regression (now retrained on
the full dataset for deployment) sits alongside it as a **cross-check, not a replacement**: where
the rule and the model agree, confidence is higher; where they disagree, that disagreement is
itself useful information for a reviewer, not noise to average away.

On top of both, a light **K-Means archetype layer** (4 clusters on traffic volume, CTR, position,
engagement, and age) groups pages into a handful of recognizable types, so the queue reads as
"here's a High-Traffic-Underperforming-CTR page" instead of a bare row of numbers.

In [1]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)

for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{col}"] = np.log1p(df[col])
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)  # only for the model fit below

# --- the Week-4/8 rule, unchanged: score, ONE reason code, an action label ---
has_position = df["position_tier"] != "no_data"
bench = (
    df[has_position].groupby("position_tier")
      .apply(lambda x: 100 * x["clicks_90d"].sum() / x["impressions_90d"].sum())
)
bench_map = bench.to_dict()

df["visible"] = (df["impressions_90d"] >= 300).astype(int)
df["bench_ctr"] = df["position_tier"].map(bench_map)
df["ctr_gap_ratio"] = np.where(
    has_position & df["bench_ctr"].gt(0),
    np.clip((df["bench_ctr"] - df["ctr"]) / df["bench_ctr"], 0, None),
    np.nan,
)

def assign_reason(row):
    if row["visible"] == 0:
        return "low_visibility"
    if row["position_tier"] == "no_data" or pd.isna(row["ctr_gap_ratio"]):
        return "no_position_data"
    if row["ctr_gap_ratio"] > 0.30:
        return "ctr_gap_high_visibility"
    elif row["ctr_gap_ratio"] > 0:
        return "ctr_gap_moderate"
    return "on_par_or_above"

df["reason_code"] = df.apply(assign_reason, axis=1)
df["rule_score"] = np.where(
    df["reason_code"].isin(["ctr_gap_high_visibility", "ctr_gap_moderate"]),
    df["ctr_gap_ratio"] * np.log1p(df["impressions_90d"]),
    0.0,
)
action_map = {"ctr_gap_high_visibility": "refresh_now", "ctr_gap_moderate": "review_soon"}
df["action"] = df["reason_code"].map(action_map).fillna("no_action")

# opportunity size: clicks/90d left on the table if this page matched its own tier's benchmark CTR
df["potential_click_gain_90d"] = np.where(
    df["bench_ctr"].notna(),
    np.clip(df["impressions_90d"] * (df["bench_ctr"] - df["ctr"]) / 100, 0, None),
    0.0,
)
print(df["action"].value_counts())


action
no_action      16852
refresh_now    10719
review_soon     2429
Name: count, dtype: int64


In [2]:
# --- archetype layer: 4-cluster K-Means, same spirit as the paper's Content Archetypes appendix ---
cluster_feats = ["log_impressions_90d", "ctr", "avg_position", "engagement_rate", "content_age_days"]
Xc = df[cluster_feats].fillna(df[cluster_feats].median())
Xs = StandardScaler().fit_transform(Xc)
km = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=10).fit(Xs)
df["archetype_id"] = km.labels_

# named AFTER inspecting the cluster centroids (median profile below) — not decided up front
archetype_names = {
    0: "Niche High-Intent (low volume, strong CTR)",
    1: "High-Traffic, Underperforming CTR",
    2: "Aging, Deep-Ranked, Low Efficiency",
    3: "Thin-Signal Long Tail",
}
df["archetype"] = df["archetype_id"].map(archetype_names)

archetype_profile = df.groupby("archetype").agg(
    n=("archetype", "size"),
    median_impressions_90d=("impressions_90d", "median"),
    median_ctr=("ctr", "median"),
    median_position=("avg_position", "median"),
    median_content_age_days=("content_age_days", "median"),
).round(2)
archetype_profile


,n,median_impressions_90d,median_ctr,median_position,median_content_age_days
archetype,,,,,
"Aging, Deep-Ranked, Low Efficiency",8740,769.0,0.04,21.5,441.0
"High-Traffic, Underperforming CTR",13488,2611.0,0.19,10.5,148.0
"Niche High-Intent (low volume, strong CTR)",163,3.0,33.33,3.6,297.0
Thin-Signal Long Tail,7609,14.0,0.00,6.9,182.0


In [3]:
# --- archetype -> typical action mapping (observed, not assumed) ---
archetype_action_mix = pd.crosstab(df["archetype"], df["action"], normalize="index").round(2)
archetype_opportunity = df.groupby("archetype")["potential_click_gain_90d"].sum().round(0).rename("total_potential_clicks_90d")
archetype_playbook = archetype_action_mix.join(archetype_opportunity).sort_values("total_potential_clicks_90d", ascending=False)
archetype_playbook


,no_action,refresh_now,review_soon,total_potential_clicks_90d
archetype,,,,
"High-Traffic, Underperforming CTR",0.38,0.49,0.13,115612.0
"Aging, Deep-Ranked, Low Efficiency",0.45,0.47,0.08,46284.0
Thin-Signal Long Tail,1.00,0.00,0.00,785.0
"Niche High-Intent (low volume, strong CTR)",1.00,0.00,0.00,0.0


**Archetype → action, read plainly:**
- **High-Traffic, Underperforming CTR** is the single biggest opportunity pool
  (~116K potential clicks/90d across the group) and the archetype most often flagged
  `refresh_now`. **Start here.**
- **Aging, Deep-Ranked, Low Efficiency** flags `refresh_now`/`review_soon` almost as often, but
  the opportunity pool is roughly a third the size — a candidate for **consolidation into a
  stronger page** rather than a rewrite, which is a strategy call for a human, not the rule.
- **Niche High-Intent** and **Thin-Signal Long Tail** are 100% `no_action` — already efficient for
  their size, or too small in absolute volume to be worth review time by default.

**The decay/refresh insight, restated for this playbook:** Week-4's signal audit found staleness
(`days_since_last_update`) does **not** reliably predict decline on its own (MIXED verdict — the
stalest pages had the *lowest* decline rate, not the highest). So this playbook does **not**
recommend refreshing by calendar age. The trigger is the **CTR gap against a page's own position
tier** — a page can be six months old and fine, or six weeks old and already underperforming.
Age only enters as a tie-breaker between equally CTR-gapped pages, never as the reason itself.

In [4]:
# --- the model, retrained on the full dataset for deployment (validation numbers live in w06) ---
NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]
preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC_FEATURES),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL_FEATURES),
])
deployed_model = Pipeline([("pre", preprocess), ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
deployed_model.fit(df[NUMERIC_FEATURES + CATEGORICAL_FEATURES], df["is_declining_label"])
df["model_decline_prob"] = deployed_model.predict_proba(df[NUMERIC_FEATURES + CATEGORICAL_FEATURES])[:, 1]

# rule x model agreement — a QA layer, not a second ranking
df["agreement"] = np.select(
    [
        (df["action"] != "no_action") & (df["model_decline_prob"] >= 0.5),
        (df["action"] != "no_action") & (df["model_decline_prob"] < 0.5),
        (df["action"] == "no_action") & (df["model_decline_prob"] >= 0.5),
    ],
    ["rule_and_model_agree", "rule_only_flag", "model_only_flag"],
    default="both_quiet",
)
print(df["agreement"].value_counts())
print()
print("Reminder: precision@50 for this model under an honest, client-grouped holdout was 0.72")
print("(measured in w05_model.ipynb / re-confirmed in w06_validation_audit.ipynb) — that number,")
print("not a train-set number, is the one this playbook's confidence claims trace back to.")


agreement
rule_and_model_agree    10640
both_quiet               8785
model_only_flag          8067
rule_only_flag           2508
Name: count, dtype: int64

Reminder: precision@50 for this model under an honest, client-grouped holdout was 0.72
(measured in w05_model.ipynb / re-confirmed in w06_validation_audit.ipynb) — that number,
not a train-set number, is the one this playbook's confidence claims trace back to.


In [5]:
# --- final ranked queue ---
queue_cols = [
    "content_id", "client_id", "rule_score", "reason_code", "action", "archetype",
    "model_decline_prob", "agreement", "potential_click_gain_90d",
    "ctr", "bench_ctr", "position_tier", "avg_position", "impressions_90d",
    "days_since_last_update", "content_type",
]
ranked_queue = (
    df[queue_cols]
      .sort_values(["rule_score", "potential_click_gain_90d"], ascending=[False, False])
      .reset_index(drop=True)
)
ranked_queue.insert(0, "rank", np.arange(1, len(ranked_queue) + 1))
ranked_queue.head(10)


,rank,content_id,client_id,rule_score,reason_code,action,archetype,model_decline_prob,agreement,potential_click_gain_90d,ctr,bench_ctr,position_tier,avg_position,impressions_90d,days_since_last_update,content_type
0,1,content_c8e9d6ab9013,client_19581e27de,12.248552,ctr_gap_high_visibility,refresh_now,"High-Traffic, Underperforming CTR",0.945957,rule_and_model_agree,731.048525,0.00,0.350324,page_1,9.7,208678,104,keyword article
1,2,content_8451fc6f034d,client_d029fa3a95,11.745546,ctr_gap_high_visibility,refresh_now,"High-Traffic, Underperforming CTR",0.573571,rule_and_model_agree,1247.741173,0.03,0.488486,top_3,2.3,272144,20,keyword article
2,3,content_4a6607efcb46,client_6208ef0f77,11.519574,ctr_gap_high_visibility,refresh_now,"High-Traffic, Underperforming CTR",0.783446,rule_and_model_agree,612.786996,0.01,0.488486,top_3,2.2,128068,104,keyword article
3,4,content_453722754fea,client_f369cb89fc,11.511711,ctr_gap_high_visibility,refresh_now,"High-Traffic, Underperforming CTR",0.898586,rule_and_model_agree,476.722059,0.01,0.350324,page_1,7.6,140079,20,keyword article
4,5,content_fb4bf6555c79,client_6208ef0f77,11.339690,ctr_gap_high_visibility,refresh_now,"Aging, Deep-Ranked, Low Efficiency",0.804971,rule_and_model_agree,130.264067,0.00,0.154905,page_3_5,45.6,84093,104,keyword article
5,6,content_39881853ef0c,client_f369cb89fc,11.298148,ctr_gap_high_visibility,refresh_now,"High-Traffic, Underperforming CTR",0.882822,rule_and_model_agree,382.639567,0.01,0.350324,page_1,7.2,112434,20,keyword article
6,7,content_c84a0ab98e90,client_f369cb89fc,11.261452,ctr_gap_high_visibility,refresh_now,"High-Traffic, Underperforming CTR",0.800706,rule_and_model_agree,715.189965,0.03,0.350324,page_1,7.8,223271,20,keyword article
7,8,content_e752a4e03dd3,client_6208ef0f77,11.021530,ctr_gap_high_visibility,refresh_now,"High-Traffic, Underperforming CTR",0.833948,rule_and_model_agree,189.668750,0.01,0.154905,page_3_5,23.9,130892,104,keyword article
8,9,content_0919dd345d80,client_4e07408562,11.021400,ctr_gap_high_visibility,refresh_now,"High-Traffic, Underperforming CTR",0.747280,rule_and_model_agree,393.802025,0.02,0.350324,page_1,7.0,119217,7,keyword article
9,10,content_54baba704595,client_6208ef0f77,11.019563,ctr_gap_high_visibility,refresh_now,"Aging, Deep-Ranked, Low Efficiency",0.716256,rule_and_model_agree,189.270262,0.01,0.154905,page_3_5,47.0,130617,104,keyword article


## 2. Intended use and limits

**Intended use:** a weekly/monthly **prioritization aid** for a content team deciding which pages
to look at first — not a scorer of content quality, not a traffic forecaster, not a client-facing
report on its own. It answers "where should a human look first," nothing stronger.

**Who it's for:** an SEO/content strategist or account lead triaging a client's content library,
with time to review maybe the top 20–50 rows in a sitting — not an automated pipeline acting on
row 4,000.

**Where it stops being valid:**
- **Content-type coverage gap.** The Week-6 audit found the validation split had zero
  `feedly article` / `comparison article` rows in its holdout by chance — this playbook's
  confidence numbers are only demonstrated for `keyword article` pages. Treat flags on the other
  two types as **unvalidated**, not wrong.
- **Single snapshot, not a time series.** Every row is one 90-day window; there's no real
  calendar date to detect seasonality, algorithm updates, or SERP feature changes over time.
- **Partial window overlap.** Per the Week-6 leakage audit, some of the model's numeric features
  structurally overlap the label's own definition window — the 0.72 precision@50 is a *soft
  upper bound*, not a leakage-free estimate.
- **Pseudonymized data.** `content_id`/`client_id` are anonymized; nothing here has been checked
  against a live site, live SERP, or a real editorial calendar.
- **No causal claim, anywhere.** Every score is directional and decision-support. Nothing in this
  playbook says a specific edit *will* recover traffic — only that a page is worth a look.

## 3. Human review + the no-go list

**What a person must check before acting on a `refresh_now`/`review_soon` row:**
1. Is a SERP feature (featured snippet, AI overview, image/video pack) sitting above this result
   and absorbing clicks that a title/meta fix can't win back? (Week-5's false-positive review
   found exactly this pattern.)
2. Is the page still correctly indexed and crawlable — a 0% CTR from a de-indexed page needs a
   technical fix, not a content rewrite.
3. Is this a seasonal or one-off dip rather than a structural decline? This dataset has no prior-
   year comparison to check that automatically — a human needs external context here.
4. If several flagged pages share one `client_id` (Week-4's top-10 review found three from a
   single client), check for a **client-wide tracking or template issue** before treating each
   page as an independent content problem.
5. Is the content still factually accurate, on-brand, and compliant? The rule never reads content
   — only traffic and position numbers.

**What should NEVER be automated:**
- **Publishing** any rewritten title, meta description, or body copy without a human editor's
  sign-off.
- **Deleting, de-indexing, or redirecting** a page ("zombie" cleanup) — these are hard to reverse
  and this playbook has no signal about legal, brand, or backlink value tied to a page.
- **Changing anything live on a client's site** directly from this notebook's output.
- **Sending client-facing reports or messages** built from these numbers without review — the
  language here is deliberately hedged (`decision-support`, `directional`) and a raw table export
  can read as a firmer promise than it is.
- **Treating a `no_action` row as "verified healthy."** It means *not flagged by this rule*, which
  is not the same as *confirmed fine* — especially for the two content types missing from
  validation.

## 4. Monitoring / retrain triggers

**What would tell us this playbook has gone stale:**
- **Precision drift.** If a fresh, client-grouped holdout's precision@50 falls meaningfully below
  the 0.72 baseline established in `w05_model.ipynb`/`w06_validation_audit.ipynb`, that's a
  retrain trigger, not a "keep shipping it" situation.
- **Reason-code / archetype mix drift.** A sudden shift in the share of `ctr_gap_high_visibility`
  rows, or in archetype sizes, more likely signals a tracking/pipeline change (e.g. a GA4/GSC
  schema update) than a genuine shift in every client's content quality at once — check the data
  pipeline before trusting the new numbers.
- **Stale CTR benchmarks.** The `bench_ctr` per position tier is computed once, from this
  snapshot. Search behavior changes (AI Overviews reshaping click patterns is the paper's own
  Finding #9) — benchmarks should be recomputed on a fixed schedule, not treated as permanent.
- **New client onboarding.** Week-6 showed how differently a client-grouped holdout can score
  versus a naive one. A new client should be treated as **unvalidated** for this playbook until a
  grouped re-check includes them, not silently folded into the existing numbers.
- **Cadence:** recompute benchmarks and re-validate at least quarterly, or immediately after any
  change to the underlying GSC/GA4 export.

In [6]:
# --- cost/value: effort tier x total opportunity, to help set the review order ---
effort_tier = {"refresh_now": "High effort (content rewrite / title+meta overhaul)",
               "review_soon": "Medium effort (light edit / metadata tweak)",
               "no_action": "No effort — not prioritized"}
cost_value = (
    df.groupby("action")
      .agg(n=("action", "size"), total_potential_clicks_90d=("potential_click_gain_90d", "sum"))
      .round(0)
)
cost_value["effort"] = cost_value.index.map(effort_tier)
cost_value = cost_value.sort_values("total_potential_clicks_90d", ascending=False)
cost_value


,n,total_potential_clicks_90d,effort
action,,,
refresh_now,10719,148758.0,High effort (content rewrite / title+meta over...
review_soon,2429,12409.0,Medium effort (light edit / metadata tweak)
no_action,16852,1514.0,No effort — not prioritized


**Reading the cost/value table:** `refresh_now` carries the highest effort per page but also
the overwhelming majority of the recoverable-click opportunity — that's the queue worth a
content team's time first. `review_soon` rows are cheaper to act on individually but each one is
worth less; batching them (e.g. a metadata-only sprint) is likely more efficient than one-by-one
review. `no_action` rows should stay off anyone's desk by default.

## 5. Exports for the paper

Writing the ranked queue, the archetype/cost-value tables, and the validation "receipts" the
paper will cite — plus two figures to `work/figures/`.

In [7]:
os.makedirs("../outputs", exist_ok=True)
os.makedirs("../figures", exist_ok=True)

# 1) the ranked queue itself (stays out of git by design — regenerated by this notebook)
queue_path = "../outputs/content_action_playbook.csv"
ranked_queue.to_csv(queue_path, index=False)
print(f"Wrote {len(ranked_queue):,} ranked rows to {queue_path}")

# 2) metrics JSON — the receipts the paper's numbers trace back to (small, committed to git)
metrics = {
    "validated_in": ["w05_model.ipynb", "w06_validation_audit.ipynb"],
    "split_design": "GroupShuffleSplit by client_id, test_size=0.2, random_state=42",
    "precision_at_50_honest_split": 0.72,
    "precision_at_50_naive_split_inflated": 0.92,
    "auc_honest_split": 0.616,
    "auc_naive_split_inflated": 0.711,
    "base_rate_test": 0.511,
    "known_limits": [
        "zero feedly/comparison-article rows in the validation holdout by chance",
        "numeric features partially overlap the label's own 30-day trend window",
        "single 90-day snapshot per row, no true time series",
    ],
    "action_counts": df["action"].value_counts().to_dict(),
    "archetype_counts": df["archetype"].value_counts().to_dict(),
}
metrics_path = "../outputs/w07_playbook_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Wrote metrics receipts to {metrics_path}")


Wrote 30,000 ranked rows to ../outputs/content_action_playbook.csv
Wrote metrics receipts to ../outputs/w07_playbook_metrics.json


In [8]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# figure 1: opportunity by archetype
fig, ax = plt.subplots(figsize=(7, 4))
archetype_opportunity.sort_values().plot(kind="barh", ax=ax, color="#3b6fa0")
ax.set_xlabel("Total potential clicks / 90d left on the table")
ax.set_title("Recoverable-click opportunity by content archetype")
fig.tight_layout()
fig.savefig("../figures/w07_opportunity_by_archetype.png", dpi=150)
plt.close(fig)

# figure 2: action mix by archetype
fig, ax = plt.subplots(figsize=(7, 4))
archetype_action_mix[["refresh_now", "review_soon", "no_action"]].plot(
    kind="barh", stacked=True, ax=ax, color=["#c0504d", "#f2b134", "#9bbb59"]
)
ax.set_xlabel("Share of pages")
ax.set_title("Action mix by content archetype")
fig.tight_layout()
fig.savefig("../figures/w07_action_mix_by_archetype.png", dpi=150)
plt.close(fig)

print("Saved 2 figures to ../figures/")
print(sorted(os.listdir("../figures")))


Saved 2 figures to ../figures/
['w07_action_mix_by_archetype.png', 'w07_opportunity_by_archetype.png']


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — only pseudonymous `content_id` / `client_id`
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.